NOME: Márcio Guilherme Ventola Pereira
RGM: 11241101717
NOME: Lucas Queroz
RGM: 11241100869

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd

pd.set_option('display.precision', 4)
print('Bibliotecas importadas com sucesso.')

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Classes:', list(iris.target_names))
print('Variáveis:', list(iris.feature_names))
print('Amostras de treino:', len(X_train))
print('Amostras de teste:', len(X_test))

In [ ]:
modelos = {
    'Linear': SVC(kernel='linear', C=1),
    'RBF': SVC(kernel='rbf', C=1, gamma='scale'),
    'Polinomial': SVC(kernel='poly', degree=3, C=1, gamma='scale')
}

resultados_kernels = []

for nome, svm in modelos.items():
    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', svm)
    ])
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados_kernels.append({
        'Kernel': nome,
        'Acurácia': accuracy_score(y_test, pred)
    })

pd.DataFrame(resultados_kernels).style.format({'Acurácia': '{:.2%}'})

### Tarefa 1 Teste do kernel polinomial

O kernel polinomial permite representar relações não lineares por meio de combinações polinomiais das características. Neste teste foi utilizado grau 3.


In [ ]:
modelo_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='poly', degree=3, C=1.0, gamma='scale'))
])

modelo_poly.fit(X_train, y_train)
y_pred_poly = modelo_poly.predict(X_test)

print(f'Acurácia: {accuracy_score(y_test, y_pred_poly):.2%}')
print('\nMatriz de confusão:')
print(confusion_matrix(y_test, y_pred_poly))
print('\nRelatório de classificação:')
print(classification_report(y_test, y_pred_poly, target_names=iris.target_names))

**Conclusão da Tarefa 1:** o kernel polinomial apresenta desempenho elevado no Iris. Eventuais erros normalmente ocorrem entre Versicolor e Virginica, espécies cujas medidas são mais semelhantes. O resultado exato pode variar conforme a divisão dos dados e a versão das bibliotecas.


## 4 Tarefa 2 Teste de diferentes valores de C

In [ ]:
valores_C = [0.1, 1, 10, 100]
resultados_C = []

for c in valores_C:
    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=c, gamma='scale'))
    ])
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados_C.append({
        'C': c,
        'Acurácia': accuracy_score(y_test, pred),
        'Matriz de confusão': confusion_matrix(y_test, pred).tolist()
    })

pd.DataFrame(resultados_C)[['C', 'Acurácia']].style.format({'Acurácia': '{:.2%}'})

In [ ]:
for resultado in resultados_C:
    print(f"C = {resultado['C']} | Acurácia = {resultado['Acurácia']:.2%}")
    print(resultado['Matriz de confusão'], '\n')

**Análise:** `C` controla a penalização dos erros. Um valor baixo aceita mais erros e favorece uma margem mais ampla; um valor alto tenta corrigir mais erros, criando uma fronteira mais ajustada. Se dois valores obtiverem a mesma acurácia, recomenda-se o menor por representar uma solução menos agressiva e com menor risco de sobreajuste.


## 5 Tarefa 3 Teste de diferentes valores de gamma

In [ ]:
valores_gamma = ['scale', 'auto', 0.1, 1]
resultados_gamma = []

for g in valores_gamma:
    modelo = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1, gamma=g))
    ])
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados_gamma.append({
        'Gamma': str(g),
        'Acurácia': accuracy_score(y_test, pred),
        'Matriz de confusão': confusion_matrix(y_test, pred).tolist()
    })

pd.DataFrame(resultados_gamma)[['Gamma', 'Acurácia']].style.format({'Acurácia': '{:.2%}'})

In [ ]:
for resultado in resultados_gamma:
    print(f"Gamma = {resultado['Gamma']} | Acurácia = {resultado['Acurácia']:.2%}")
    print(resultado['Matriz de confusão'], '\n')

**Análise:** `gamma` determina o alcance da influência de cada amostra no kernel RBF. Valores pequenos criam uma fronteira mais suave; valores altos tornam a influência mais localizada e podem gerar uma fronteira complexa. No Iris, as diferenças costumam ser pequenas por se tratar de um conjunto simples e bem organizado.


## 6 Validação cruzada dos modelos

In [ ]:
resultados_cv = []

for nome, svm in modelos.items():
    modelo = Pipeline([('scaler', StandardScaler()), ('svm', svm)])
    scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
    resultados_cv.append({
        'Kernel': nome,
        'Média': scores.mean(),
        'Desvio padrão': scores.std()
    })

pd.DataFrame(resultados_cv).style.format({
    'Média': '{:.2%}',
    'Desvio padrão': '{:.4f}'
})

# 7 Questões de fixação

### 1 O que caracteriza um problema de classificação supervisionada

É um problema no qual os exemplos de treinamento possuem classes conhecidas. O algoritmo aprende a relação entre as variáveis de entrada e os rótulos para classificar novas amostras.

### 2 Quais são as variáveis de entrada e a variável alvo no Iris

As entradas são comprimento e largura da sépala, além de comprimento e largura da pétala. A variável alvo é a espécie da flor: Setosa, Versicolor ou Virginica.

### 3 Qual é a ideia principal do SVM ao construir uma fronteira de decisão

O SVM procura uma fronteira que separe as classes com a maior margem possível. Uma margem ampla tende a melhorar a generalização para dados novos.

### 4 O que são vetores de suporte

São as amostras mais próximas da fronteira de decisão. Elas têm papel direto na definição da posição e da margem do hiperplano.

### 5 Qual é a diferença entre um SVM linear e um não linear

O SVM linear utiliza uma fronteira reta ou plana. O não linear usa kernels, como RBF e polinomial, para representar fronteiras curvas e relações mais complexas.

### 6 Por que a padronização é importante para SVM

Porque o método utiliza distâncias e produtos entre as características. Sem padronização, variáveis com valores numericamente maiores podem dominar o treinamento. O StandardScaler deixa as escalas comparáveis.

### 7 O que significa o parâmetro C em termos intuitivos

C controla o equilíbrio entre uma margem ampla e a penalização de erros. C baixo permite mais erros e favorece uma fronteira simples; C alto penaliza mais os erros e ajusta a fronteira aos dados.

### 8 Qual é o papel de gamma no kernel RBF

Gamma define a extensão da influência de cada exemplo. Gamma baixo produz influência ampla e fronteira suave; gamma alto produz influência localizada e pode criar uma fronteira mais detalhada.

### 9 O SVM linear teve desempenho satisfatório Como justificar

Sim. O código mostra uma acurácia elevada no conjunto de teste. Além disso, a validação cruzada permite confirmar que o resultado não depende somente de uma divisão específica.

### 10 O kernel RBF melhorou o resultado em relação ao linear

A resposta deve ser verificada na tabela executada acima. Na divisão utilizada, compare diretamente as acurácias. Mesmo quando há melhora, ela pode ser pequena, devendo-se considerar também a simplicidade do modelo linear.

### 11 Em quais espécies ocorreram mais confusões

As confusões tendem a ocorrer entre Versicolor e Virginica, pois apresentam características mais parecidas. A Setosa geralmente é separada com facilidade.

### 12 O que a validação cruzada acrescenta

Ela avalia o modelo em diferentes divisões dos dados, oferecendo uma estimativa mais confiável do desempenho e reduzindo a dependência de um único conjunto de teste.

### 13 Por que escolher o modelo mais simples quando os desempenhos são parecidos

Porque ele costuma ser mais fácil de interpretar, mais rápido e menos sujeito a sobreajuste. Complexidade adicional só se justifica quando traz uma melhoria consistente.

### 14 O que aconteceu ao testar o kernel polinomial

Ele apresentou bom desempenho e classificou corretamente a maior parte das amostras. Entretanto, deve-se comparar sua acurácia e validação cruzada com os kernels linear e RBF antes de afirmar que é a melhor opção.

### 15 Qual valor de C pareceu mais adequado

Deve-se escolher o menor valor entre os que alcançaram o melhor desempenho consistente. Nos testes, C igual a 1 costuma oferecer bom equilíbrio entre acurácia e simplicidade, mas a tabela executada contém a resposta para esta divisão.

### 16 As alterações em gamma mudaram muito o resultado

No Iris, normalmente não. Os valores testados produzem resultados próximos, indicando baixa sensibilidade neste experimento. Em conjuntos mais complexos, gamma pode afetar bastante a fronteira.

### 17 Quando usar um kernel não linear

Quando uma linha ou um plano não separa adequadamente as classes e há evidências de relações mais complexas. Se o linear tiver desempenho equivalente, ele normalmente é preferível por ser mais simples.

### 18 Vantagens e limitações do Iris para ensino

O Iris é pequeno, organizado, balanceado e fácil de compreender. Permite treinar modelos rapidamente e visualizar conceitos como classificação, acurácia e matriz de confusão. Porém, é muito mais limpo e simples do que problemas reais, que podem conter ruído, valores ausentes, milhares de variáveis, classes desbalanceadas e grande volume de dados.